# YorTIB-200 — Paired Statistical Analysis

Computes the paired correctness table, exact McNemar test, accuracy difference, and bootstrap confidence interval.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("../data/yortib_200.json")
with open(DATA, encoding="utf-8") as f:
    data = json.load(f)
if isinstance(data, dict):
    data = data.get("data", data.get("items", []))
df = pd.DataFrame(data)
print("Rows:", len(df))
df.head()


In [ ]:
baseline = read_prediction(Path("../predictions/baseline.csv"))
sovereign = read_prediction(Path("../predictions/sovereign.csv"))

df["baseline_correct"] = baseline.values == df["answer"].astype(str).str.strip().values
df["sovereign_correct"] = sovereign.values == df["answer"].astype(str).str.strip().values

both_correct = int((df.baseline_correct & df.sovereign_correct).sum())
baseline_only = int((df.baseline_correct & ~df.sovereign_correct).sum())
sovereign_only = int((~df.baseline_correct & df.sovereign_correct).sum())
both_wrong = int((~df.baseline_correct & ~df.sovereign_correct).sum())

contingency = pd.DataFrame(
    [[both_correct, baseline_only],
     [sovereign_only, both_wrong]],
    index=["Baseline correct","Baseline incorrect"],
    columns=["Sovereign correct","Sovereign incorrect"]
)
display(contingency)


In [ ]:
from scipy.stats import binomtest

n_discordant = baseline_only + sovereign_only
mcnemar_p = binomtest(min(baseline_only, sovereign_only),
                       n=n_discordant, p=0.5).pvalue

baseline_acc = df.baseline_correct.mean()
sovereign_acc = df.sovereign_correct.mean()
difference = sovereign_acc - baseline_acc

print(f"Baseline accuracy: {baseline_acc:.3f}")
print(f"Sovereignty-preserving accuracy: {sovereign_acc:.3f}")
print(f"Absolute difference: {difference:.3f} ({difference*100:.1f} percentage points)")
print(f"Exact McNemar p: {mcnemar_p:.3e}")


In [ ]:
rng = np.random.default_rng(20260929)
paired_difference = (
    df.sovereign_correct.astype(int) -
    df.baseline_correct.astype(int)
).to_numpy()

B = 10000
boot = np.array([
    rng.choice(paired_difference, size=len(paired_difference), replace=True).mean()
    for _ in range(B)
])

ci = np.quantile(boot, [0.025, 0.975])
print("Bootstrap estimate:", paired_difference.mean())
print("95% bootstrap CI:", ci)


In [ ]:
errors = df.loc[~df.sovereign_correct,
    ["id","focus_word","answer"]]
display(errors)
